# Prepare the shared model splits

Run this section before the weighted baseline and advanced models. It exports train, validation, and test once, along with their manifest.

Run from the repository root.

# 数据准备：64% 训练 / 16% 验证 / 20% 测试
保留最新 clean_data.ipynb 的清洗顺序，修复 stemming 丢失 rating 的问题。
先按原流程划分 80% development / 20% test，再将 development 按 80% / 20% 划为 train / validation。
Spark randomSplit 的比例是期望比例，实际行数可能略有偏差。所有组合必须共用一次导出的数据。
设置下面两个路径；不要覆盖上一轮数据。测试集不参与 embedding 训练、选轮次或调参。


In [ ]:
from pathlib import Path
RAW_CSV = Path.cwd() / 'drugsComTrain_raw.csv'
OUTPUT_DIR = RAW_CSV.parent / 'data_64_16_20'
assert RAW_CSV.is_file(), RAW_CSV


In [ ]:

import os
import sys

# 设置环境变量
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

# 创建 SparkSession
spark = (
    SparkSession.builder
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.execution.pythonUDF.arrow.enabled", "false")
    .config("spark.python.worker.reuse", "false")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .getOrCreate()
)
#初始化
ss = spark

sc = ss.sparkContext

ss.sparkContext.setLogLevel("ERROR")
# from pyspark.ml.feature import CountVectorizer, IDF, Tokenizer


### 数据清洗全步骤
- 删除缺失值
- 无效值处理（不存在）
- 删除重复数据
- 评分冲突处理
- 数据标准化
- 词干提取
- 字符串分词
- 单引号连接词分词
- 删除停用词

In [ ]:
# 转化为dataframe
data = ss.read.csv(path=str(RAW_CSV),
                        sep=',',
                        encoding='UTF-8',
                        comment=None,
                        header=True,
                        inferSchema=True,
                        quote='"',        # 明确指定包裹文本的引号是双引号
                        escape='"',       #内部引号用 "" 转义
                        multiLine=True    #允许一个字段跨多行
                        )


data.select('usefulCount').sort('usefulCount') #数据正常 最小值=0
data.show(20, truncate=100)
 

In [ ]:
#只保留review rating 
df_data = data.select('review','rating').toDF('review','rating')

from pyspark.sql.functions import col, count, trim

# 按 review、rating 分组，统计每组条数
dup_group = df_data.groupBy("review", "rating")\
    .agg(count("*").alias("cnt"))\
    .filter(col("cnt") >= 2)  # 保留重复的组合

dup_group.show() 
# 有多少组重复的(review,rating)组合
dup_group.count() #48652

# 去重处理
df_data = df_data.distinct()
df_data.count()


In [ ]:
from pyspark.sql import functions as f

# 标签冲突：相同review，rating不同
#1. 统计rating不止一个的review
conflict_reviews = (
    df_data.groupBy("review")
    .agg(
        f.countDistinct("rating").alias("different_ratings")
    )
    .filter(col("different_ratings") > 1)
)

conflict_records = df_data.join(
    conflict_reviews.select("review"),
    on="review",
    how="inner"
)

print("涉及评分冲突的行数：", conflict_records.count())

conflict_reviews.sort("different_ratings", ascending=False).show(20)
# 最多重复数据有6条，最少两条




In [ ]:
# 2.对冲突review：合并成一行，rating取均值四舍五入，转化为int
conflict_rating = (
    df_data.join(conflict_reviews.select("review"), on="review", how="inner")
    .groupBy("review")
    .agg(
        f.round(f.avg("rating")).cast("int").alias("rating")
    )
)
conflict_rating.show(5)
conflict_rating.count()

# 3. 取出【没有冲突的正常数据】
normal_data = df_data.join(
    conflict_reviews.select("review"),
    on="review",
    how="leftanti"   # leftanti：只保留左边匹配不到的行（非冲突）
)
# 4. 拼接：正常数据 + 合并后的冲突单行数据
df_clean = normal_data.unionByName(conflict_rating)
df_clean.count()


In [ ]:
#缺失值处理
df_clean = df_clean.filter(
    (col("review").isNotNull()) & (trim(col("review")) != "") &
    (col("rating").isNotNull()) & (trim(col("rating")) != "")
)
df_clean.count()

In [ ]:
# 检查数据是否有效
df_clean = df_clean.filter(
    col("rating").between(1, 10))

df_clean.count() 

In [ ]:
# 标准化处理
import html
import re
import unicodedata
from pyspark.sql.types import StringType

def normalize_review(text):
    if text is None:
        return None
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)
    # 删除常见 HTML 标签
    text = re.sub(r"</?[A-Za-z][^>]*>", " ", text)
    # 统一弯引号
    text = text.replace("’", "'").replace("‘", "'")
    # 合并空格、换行和制表符
    text = re.sub(r"\s+", " ", text).strip()
    return text

# 单独一行注册UDF
normalize_review = f.udf(normalize_review, returnType=StringType())

df_clean = df_clean.withColumn(
    "review",
    normalize_review(col("review"))
)

# 小写；保留字母、数字、空白和英文单引号
# 保留单引号，避免把 don't 直接破坏掉
df_norm = df_clean.withColumn(
    "review",
    trim(
        f.regexp_replace(
            f.regexp_replace(
                f.lower(col("review")),
                r"[^\p{L}\p{N}\s']",
                " "
            ),
            r"\s+",
            " "
        )
    )
)
df_norm.show(20, truncate=100)

In [ ]:
from nltk.stem import PorterStemmer

# 词干提取需要使用此包
ps = PorterStemmer()

def stemming(data_str):
    # 期望输入为字符串
    cleaned_str = ''
    text = data_str.split()

    for word in text:
        cleaned_str+= ps.stem(word) + ' '

    return cleaned_str

#定义 UDF 函数
sent_to_stem_words_udf = f.udf(lambda x: str(stemming(x)), StringType())


df_stem = df_norm.select('review', 'rating', sent_to_stem_words_udf(df_norm.review).alias('stemmed_review'))
df_stem.show(10, truncate=100)

In [ ]:
# 分词处理
from pyspark.ml.feature import  Tokenizer
tokenizer = Tokenizer(inputCol="stemmed_review", outputCol="words") # Tokenizer 按空格将每篇文档拆分为单词
wordsData = tokenizer.transform(df_stem)
wordsData.show(10, truncate=50)

In [ ]:
#查看含义单引号的word

result = (
    wordsData
    .select(
        "review",
        "rating",
        f.explode("words").alias("token")
    )
    .filter(col("token").contains("'"))
)

# 查看所有包含单引号的唯一 token
result.select("token").distinct().show(10, truncate=False)



In [ ]:
from pyspark.sql.types import ArrayType
#展开单引号连接词
SPECIAL_FORMS = {
    "can't": ["can", "not"],
    "cannot": ["can", "not"],
    "won't": ["will", "not"],
    "shan't": ["shall", "not"],
    # ain't 的具体助动词取决于语境，这里只保留否定信息
    "ain't": ["not"]
}

#展开函数
def normalize_tokens(tokens):
    if tokens is None:
        return []

    output = []

    for token in tokens:
        if not token:
            continue

        word = token.lower().replace("’", "'").replace("‘", "'")

        # 去掉包裹词语的引号，例如 'dopey' → dopey
        word = word.strip("'\"")

        if not word:
            continue

        # 特殊否定形式
        if word in SPECIAL_FORMS:
            output.extend(SPECIAL_FORMS[word])

        # don't → do not；wasn't → was not
        elif word.endswith("n't"):
            base = word[:-3]
            if base:
                output.append(base)
            output.append("not")

        # i'm → i am
        elif word.endswith("'m"):
            output.extend([word[:-2], "am"])

        # you're → you are
        elif word.endswith("'re"):
            output.extend([word[:-3], "are"])

        # i've → i have
        elif word.endswith("'ve"):
            output.extend([word[:-3], "have"])

        # you'll → you will
        elif word.endswith("'ll"):
            output.extend([word[:-3], "will"])

        # 'd 可能是 had 或 would，先保留主体
        elif word.endswith("'d"):
            output.append(word[:-2])

        # 's 可能是所有格、is 或 has，先保留主体
        # tourette's → tourette；it's → it
        elif word.endswith("'s"):
            output.append(word[:-2])

        else:
            # 其他带单引号的词不直接删除
            output.append(word)

    return [word for word in output if word]

#这里返回array<string>
normalize_tokens = f.udf(normalize_tokens, returnType=ArrayType(StringType()))

wordsData = wordsData.withColumn(
    "normalized_words",
    normalize_tokens(col("words"))
)

wordsData.select(
    "words", "normalized_words"
).show(20, truncate=150)


In [ ]:
#停用词处理
from pyspark.ml.feature import StopWordsRemover

# 需要保留的否定词
protected_words = {
    "not", "no", "nor", "never", "neither", "without"
}

default_stopwords = StopWordsRemover.loadDefaultStopWords("english")

custom_stopwords = [
    word for word in default_stopwords
    if word.lower() not in protected_words
]

remover = StopWordsRemover(
    inputCol="normalized_words",
    outputCol="filtered_words",
    stopWords=custom_stopwords,
    caseSensitive=False
)

cleanData = remover.transform(wordsData)

cleanData.select(
    "filtered_words",
    "rating"
).show(20, truncate=150)



In [ ]:
# 80% 训练集，20% 测试集
train_data, test_data = cleanData.randomSplit(
    [0.8, 0.2],
    seed=42
)

# 缓存，方便后续多次使用
train_data = train_data.cache()
test_data = test_data.cache()

print("训练集记录数：", train_data.count())
print("测试集记录数：", test_data.count())


In [ ]:
# Export all three partitions once; model notebooks never split data again.
import json, hashlib
from collections import Counter

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if (OUTPUT_DIR / 'manifest.json').exists():
    raise FileExistsError('Inputs already exported. Use a NEW OUTPUT_DIR to avoid mixing experiments.')
development_data = train_data
train_data, validation_data = development_data.randomSplit([0.8, 0.2], seed=43)
train_data.cache(); validation_data.cache()
split_info, text_groups = {}, {}
for split_name, frame in [('train', train_data), ('validation', validation_data), ('test', test_data)]:
    target = OUTPUT_DIR / (split_name + '.jsonl')
    temporary = target.with_suffix('.jsonl.tmp')
    counts, groups, n, empty = Counter(), set(), 0, 0
    with temporary.open('w', encoding='utf-8', newline='\n') as stream:
        for row in frame.select('review','rating','filtered_words').toLocalIterator():
            words, rating = list(row.filtered_words), int(row.rating)
            assert 1 <= rating <= 10 and all(isinstance(w,str) for w in words)
            group = hashlib.sha256(json.dumps(words,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()
            record = dict(row_id=split_name+'_'+str(n), review=row.review,
                          rating=rating, filtered_words=words, text_group_id=group)
            stream.write(json.dumps(record,ensure_ascii=False)+'\n')
            counts[rating] += 1; groups.add(group); n += 1; empty += int(not words)
    temporary.replace(target)
    split_info[split_name] = dict(file=target.name,n=n,rating_counts=dict(counts),empty_token_rows=empty,
                                  sha256=hashlib.sha256(target.read_bytes()).hexdigest())
    text_groups[split_name] = groups
overlap = {a+'_'+b:len(text_groups[a] & text_groups[b]) for a,b in [('train','validation'),('train','test'),('validation','test')]}
manifest = dict(pipeline='latest_clean_data_64_16_20_v2',source_notebook_sha256='12ce8f1a29e9f205ebd96fbd5d0265d9a8fc987e8c9c93b5e7f42232e035f3c1',
                raw_csv=str(RAW_CSV.resolve()),raw_sha256=hashlib.sha256(RAW_CSV.read_bytes()).hexdigest(),
                modifications=['Preserve rating in df_stem','Keep original test_data; subdivide original train_data with seed 43','Export fixed partitions'],
                split_method='Spark 80/20 seed42, then development 80/20 seed43; expected 64/16/20',
                independent_test_available=True,split_info=split_info,normalized_text_group_overlap=overlap)
(OUTPUT_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(manifest,indent=2,ensure_ascii=False))
if any(overlap.values()):
    print('NOTICE: identical filtered text occurs across partitions. Preserved the supplied randomSplit; disclose this in the report. These are not group-isolated splits.')
print('Use this same data directory for every combination. Do not run randomSplit separately in each model notebook.')
